# NB10 — EVALUATION3 overlap audit

Notebook này chỉ phát hiện overlap giữa ảnh EVALUATION3 và các item từng được đưa vào scorer. Nó chưa chạy metric EVALUATION3 và không cần mapping `Cmt` 1/2/3.

## 1. Runtime portable

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass  # Cho phép chạy notebook ngoài Colab.

REPO_URL = 'https://github.com/ThinhTran2208/opisoverated.git'
BRANCH = 'feat/evaluation3-overlap-audit'
DEFAULT_REPO_DIR = Path('/content/opisoverated-e3-audit')
explicit_root = os.environ.get('FASHION_PROJECT_ROOT')
REPO_ROOT = Path(explicit_root).expanduser().resolve() if explicit_root else DEFAULT_REPO_DIR

def run_git(*args, cwd=None):
    return subprocess.run(
        ['git', '-c', 'http.version=HTTP/1.1', *args],
        cwd=cwd, check=True, text=True,
    )

if not (REPO_ROOT / '.git').is_dir():
    if REPO_ROOT.exists():
        raise FileExistsError(
            f'{REPO_ROOT} đã tồn tại nhưng không phải Git repo. '
            'Hãy đổi DEFAULT_REPO_DIR hoặc xóa folder lỗi rồi chạy lại.'
        )
    run_git('clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT))
else:
    run_git('fetch', 'origin', BRANCH, cwd=REPO_ROOT)
    run_git('switch', BRANCH, cwd=REPO_ROOT)
    run_git('pull', '--ff-only', 'origin', BRANCH, cwd=REPO_ROOT)

if not (REPO_ROOT / 'src/evaluation/evaluation3_overlap.py').is_file():
    raise FileNotFoundError(f'Branch {BRANCH} thiếu module overlap audit.')
os.environ['FASHION_PROJECT_ROOT'] = str(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements-evaluation.txt')], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.runtime_paths import load_runtime_paths
from src.evaluation.evaluation3_overlap import run_overlap_audit
RUNTIME_PATHS = load_runtime_paths(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)
print('ARTIFACT_ROOT:', RUNTIME_PATHS.artifact_root)
print('Runtime setup: OK')

## 2. Kiểm tra input trên Google Drive

Notebook dùng trực tiếp `MyDrive/EVALUATION3` và `MyDrive/scorer_ready_v2`; không cần sửa path thủ công. `E3_ROOT` phải chứa các folder outfit ID, mỗi folder có ảnh `U`, `B`, `S`, `G`. Không lọc `A-Test2000` vì workbook hiện tại chỉ có 30 dòng mang tag đó, không phải 2.000 dòng.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
E3_DIR = DRIVE_ROOT / 'EVALUATION3'
DRIVE_E3_ROOT = E3_DIR / 'outfit'
E3_ARCHIVE = E3_DIR / 'outfit.zip'
CMT_FILE = E3_DIR / 'Cmt_ALL_20190325.xlsx'
ATTRIBUTE_FILE = E3_DIR / 'Attribute_ALL_UBSGsimple.xlsx'
SCORER_DIR = DRIVE_ROOT / 'scorer_ready_v2'
OUTPUT_DIR = DRIVE_ROOT / 'evaluation3_overlap_audit'
LOCAL_E3_DIR = Path('/content/evaluation3_outfit')
image_suffixes = {'.jpg', '.jpeg', '.png', '.webp'}

def contains_image(root):
    return root.is_dir() and any(
        path.is_file() and path.suffix.lower() in image_suffixes
        for path in root.rglob('*')
    )

if contains_image(DRIVE_E3_ROOT):
    E3_ROOT = DRIVE_E3_ROOT
elif E3_ARCHIVE.is_file():
    if not contains_image(LOCAL_E3_DIR):
        import shutil
        print(f'Đang giải nén {E3_ARCHIVE.name} vào Colab local disk...')
        shutil.unpack_archive(str(E3_ARCHIVE), str(LOCAL_E3_DIR), format='zip')
    nested_outfit = LOCAL_E3_DIR / 'outfit'
    E3_ROOT = nested_outfit if contains_image(nested_outfit) else LOCAL_E3_DIR
else:
    E3_ROOT = DRIVE_E3_ROOT

required_files = {
    'Cmt workbook': CMT_FILE,
    'Attribute workbook': ATTRIBUTE_FILE,
    'Scorer train': SCORER_DIR / 'scorer_ready_v2_train.jsonl',
    'Scorer valid': SCORER_DIR / 'scorer_ready_v2_valid.jsonl',
}
missing = [f'{name}: {path}' for name, path in required_files.items() if not path.is_file()]
has_e3_images = contains_image(E3_ROOT)
if not has_e3_images:
    missing.append(
        f'EVALUATION3 images: cần {DRIVE_E3_ROOT} hoặc file {E3_ARCHIVE}'
    )

INPUTS_READY = not missing
if INPUTS_READY:
    print('INPUTS: OK — có thể chạy audit.')
else:
    print('INPUTS: CHƯA ĐỦ — notebook dừng an toàn, chưa chạy audit.')
    for item in missing:
        print('  -', item)

TEST_FILE = SCORER_DIR / 'scorer_ready_v2_test.jsonl'
print('E3 image source:', E3_ROOT)
print('Scorer test (tùy chọn cho strict-clean):', 'có' if TEST_FILE.is_file() else 'chưa có')

## 3. Chạy audit

Mặc định lấy ảnh Polyvore trực tiếp từ `codewaly/polyvore1000`. Nếu đã có thư mục ảnh local theo `item_id`, đặt `POLYVORE_IMAGE_ROOT` thành đường dẫn đó để tránh tải lại.

In [ ]:
summary = None
output_paths = {}
if not INPUTS_READY:
    print('SKIPPED: bổ sung các input được liệt kê ở cell trên rồi chạy lại cell 2–4.')
else:
    POLYVORE_IMAGE_ROOT = None
    POLYVORE_HF_DATASET = None if POLYVORE_IMAGE_ROOT else 'codewaly/polyvore1000'
    development_splits = {
        'train': SCORER_DIR / 'scorer_ready_v2_train.jsonl',
        'valid': SCORER_DIR / 'scorer_ready_v2_valid.jsonl',
    }
    if TEST_FILE.is_file():
        development_splits['test'] = TEST_FILE

    summary, output_paths = run_overlap_audit(
        evaluation3_root=E3_ROOT,
        development_split_paths=development_splits,
        output_dir=OUTPUT_DIR,
        polyvore_image_root=POLYVORE_IMAGE_ROOT,
        polyvore_hf_dataset=POLYVORE_HF_DATASET,
        annotations_path=CMT_FILE,
        annotation_sheet='CMT',
        metadata_path=ATTRIBUTE_FILE,
        metadata_sheet='Num',
        model_development_splits={'train', 'valid'},
        near_hamming_threshold=4,
    )
    print('AUDIT: HOÀN TẤT')

## 4. Đọc kết quả

Chỉ dùng manifest clean khi `status = PASS`. `strict_clean` loại overlap với cả Polyvore test; `model_clean` chỉ loại overlap với train/valid.

In [ ]:
import json
if summary is None:
    print('Chưa có kết quả vì input chưa đủ; xem cell 2.')
else:
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    for name, path in output_paths.items():
        print(f'{name}: {path}')